# 56. 散点图（scatterplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 13 / 20 步：探索变量关系与趋势**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 累积分布图（ecdfplot）  →  **本章任务：** 散点图（scatterplot）  →  **下一步：** 统计折线图（lineplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

面对一长串销售额和访问量的数字，光看表格很难看出它们是否存在关联。



## 本章目标

学完本章，你将能够：

- **理解**：理解「散点图（scatterplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「散点图（scatterplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「散点图（scatterplot）」并读出其中的结论。


## 56.1 适用场景
**背景引入**：面对一长串销售额和访问量的数字，光看表格很难看出它们是否存在关联。散点图把每个样本的横纵坐标同时画出来，让“关系”以点的形状直接呈现——有无趋势、是否聚成一团、有没有离群的异常点，一眼就能判断。若还想同时看一个分类或第三个维度，也能用点的颜色或大小叠加上去。

打个比方：scatterplot 像'在坐标纸上贴图钉'——每个样本的一对数值（X、Y）就是一粒图钉的位置，把几千粒图钉贴出来，'访问量高时销售额是否也高'这种关系就以点的走向直接呈现。还想多看一个维度，就让点的颜色或大小再'多记一样东西'。

检查两个数值变量关系，并同时观察分类或第三个数值变量。


## 56.2 数据结构

每行一条观察，至少两列数值，可增加类别和大小字段。


## 56.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 sizes=(20, 180) 改为 sizes=(40, 120)，观察气泡大小范围对可读性的影响
2. 调整 alpha 参数（如 0.4 或 0.9），说明透明度对重叠点显示的作用
3. 移除 style="channel" 参数，对比形状映射与纯颜色映射的信息密度


## 56.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.scatterplot()`、`ax.set()`、`ax.legend()` | 检查两个数值变量关系，并同时观察分类或第三个数值变量。 | 同时使用过多映射 |
| 进阶变体 | `plt.subplots()`、`sns.scatterplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 点大小范围过大 |
| 关键参数 | `hue` | 颜色 | 同时使用过多映射 |
| 关键参数 | `size` | 点面积 | 点大小范围过大 |
| 关键参数 | `style` | 点形 | 图例覆盖数据 |
| 关键参数 | `alpha` | 透明度 | 同时使用过多映射 |


## 56.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-56 -->
### 数学推导｜散点关系与 Pearson 相关系数

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先去掉量纲。** 标准化后 $z_{xi}=(x_i-\bar{x})/s_x$、$z_{yi}=(y_i-\bar{y})/s_y$。

**第 2 步｜看同一观测上的方向是否一致。** 当两个标准化值同号时，乘积 $z_{xi}z_{yi}$ 为正；异号时为负。

**第 3 步｜对共同变化求平均。** 样本相关可写为

$$
r=\frac{1}{n-1}\sum_{i=1}^{n}z_{xi}z_{yi}
$$

展开标准化定义，就得到分子为离差乘积、分母为两个平方和平方根的常见形式。

**把上面的关系收束为本章计算式：**

$$
r=\frac{\sum_i(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_i(x_i-\bar{x})^2\sum_i(y_i-\bar{y})^2}}
$$

**符号解释：** $r\in[-1,1]$ 描述线性共同变化的方向和强度。

**代码对应：** `df[[x, y]].corr().iloc[0, 1]` 与散点图配合使用。

**使用边界：** 相关不等于因果；异常值、非线性和分组结构都可能改变总体相关。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 56.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    alpha=0.7,
    palette="colorblind",
    ax=ax,
)
ax.set(title="访问量与销售额", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


**练一练 50.4**：在基础散点图里，把透明度参数 `alpha=0.7` 改小再画一次，观察图形发生了什么。散点重叠时，较低的 alpha 会把重叠程度变成颜色的深浅密度，从而更容易看出数据集中聚集的区域。请把 `alpha` 改为 `0.25` 重画一遍，并完成下面的自检。


In [ ]:
# 请在下方填写代码
# 任务：把基础散点图的 alpha 参数改为 0.25，观察重叠点的深浅变化。
alpha = None  # 请在这一行填入透明度数值


In [ ]:
# 请在下方填写代码
# 参考答案：把透明度降到 0.25，重叠较多的区域颜色更深、显得更“团”。
alpha = 0.25

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    alpha=alpha,
    palette="colorblind",
    ax=ax,
)
ax.set(title="访问量与销售额（半透明）", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 56.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    size="ad_spend",
    sizes=(20, 180),
    style="channel",
    alpha=0.7,
    palette="colorblind",
    ax=ax,
)
ax.set(title="访问量、销售额与广告投入", xlabel="访问量", ylabel="销售额")
ax.legend(
    title="渠道 / 广告投入",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
fig.tight_layout()
plt.show()


## 56.8 参数说明

- hue：颜色
- size：点面积
- style：点形
- alpha：透明度


## 56.9 结果解读

先看整体关系，再比较颜色组、点大小和异常观察。


## 56.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 56.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 56.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 56.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 56.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 56.12 易错点提醒

- 同时使用过多映射
- 点大小范围过大
- 图例覆盖数据


## 56.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 56.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：用 size 编码第三个变量，把散点升级为气泡
# 【目标】用点的大小再编码一列，让一个图承载三个变量。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 size="conversion"，点的大小编码转化率。
fig, ax = plt.subplots(figsize=(8, 4.6))
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    size="conversion",
    sizes=(20, 200),
    alpha=0.7,
    ax=ax,
)
ax.set(title="访问量、销售额与转化率", xlabel="访问量", ylabel="销售额")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：加上大小编码，读图时多了哪一层信息 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.8))
sns.scatterplot(
    data=orders,
    x="items",
    y="order_value",
    hue="category",
    style="satisfied",
    alpha=0.65,
    palette="colorblind",
    ax=ax,
)
ax.set(title="购买件数与客单价", xlabel="购买件数", ylabel="客单价（元）")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()


## 56.15 小结

使用scatterplot把位置、颜色、大小和样式映射到DataFrame列。


### 56.15.1 你已经掌握

- 判断散点图（scatterplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 56.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `hue` | 颜色 |
| `size` | 点面积 |
| `style` | 点形 |
| `alpha` | 透明度 |


### 56.15.3 需要注意

- 同时使用过多映射
- 点大小范围过大
- 图例覆盖数据


### 56.15.4 完成检查

- [ ] 能判断什么问题适合使用散点图（scatterplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 56.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
